# Module 03: Modeling - Logistic Regression

Linear probability classification with class weighting and threshold tuning for fraud detection.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report
)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Read dataset
try:
    credit_card_data = pd.read_csv("../creditcard.csv")
    print("Loaded ../creditcard.csv successfully.")
except FileNotFoundError:
    credit_card_data = pd.read_csv("creditcard.csv")
    print("Loaded creditcard.csv successfully.")

In [ ]:
# first 5 rows of the dataset
credit_card_data.head()

In [ ]:
credit_card_data.tail()

In [ ]:
# Preprocessing: Scale Amount and drop Time
df = credit_card_data.copy()
scaler = StandardScaler()
df['Amount_Scaled'] = scaler.fit_transform(df[['Amount']])
df.drop(columns=['Time', 'Amount'], inplace=True)

X = df.drop(columns=['Class'])
y = df['Class']

print(f"Features: {X.shape[1]}, Records: {X.shape[0]:,}")

In [ ]:
# Train-Test Split with Stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]:,} rows, Test: {X_test.shape[0]:,} rows")
print(f"Train Frauds: {y_train.sum()}, Test Frauds: {y_test.sum()}")

## Baseline Logistic Regression (Unweighted)

In [ ]:
# Train unweighted baseline
lr_baseline = LogisticRegression(max_iter=1000, random_state=42)
lr_baseline.fit(X_train, y_train)

y_pred_base = lr_baseline.predict(X_test)

print("Baseline Logistic Regression:")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_base):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_base):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_base):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_base):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, lr_baseline.predict_proba(X_test)[:, 1]):.4f}")

## Balanced Logistic Regression (class_weight='balanced')

In [ ]:
# Train balanced logistic regression
lr_balanced = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_balanced.fit(X_train, y_train)

y_pred_bal = lr_balanced.predict(X_test)
y_prob_bal = lr_balanced.predict_proba(X_test)[:, 1]

print("Balanced Logistic Regression:")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_bal):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bal):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_bal):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_bal):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_bal):.4f}")

In [ ]:
# Classification Report
print("Classification Report (Balanced):\n")
print(classification_report(y_test, y_pred_bal, target_names=['Normal', 'Fraud'], digits=4))

In [ ]:
# Confusion Matrix Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_base = confusion_matrix(y_test, y_pred_base)
cm_bal = confusion_matrix(y_test, y_pred_bal)

sns.heatmap(cm_base, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False)
axes[0].set_title(f'Baseline LR (Recall: {recall_score(y_test, y_pred_base):.2%})')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_xticklabels(['Normal', 'Fraud'])
axes[0].set_yticklabels(['Normal', 'Fraud'])

sns.heatmap(cm_bal, annot=True, fmt='d', cmap='Greens', ax=axes[1], cbar=False)
axes[1].set_title(f'Balanced LR (Recall: {recall_score(y_test, y_pred_bal):.2%})')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_xticklabels(['Normal', 'Fraud'])
axes[1].set_yticklabels(['Normal', 'Fraud'])

plt.tight_layout()
plt.show()

## ROC and Precision-Recall Curves

In [ ]:
# Plot ROC and PR Curves
fpr, tpr, _ = roc_curve(y_test, y_prob_bal)
precision, recall, _ = precision_recall_curve(y_test, y_prob_bal)
ap_score = average_precision_score(y_test, y_prob_bal)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
axes[0].plot(fpr, tpr, color='#2b5c8f', lw=2, label=f'ROC curve (AUC = {roc_auc_score(y_test, y_prob_bal):.4f})')
axes[0].plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
axes[0].set_title('Receiver Operating Characteristic (ROC)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right')

# PR Curve
axes[1].plot(recall, precision, color='#d9534f', lw=2, label=f'PR curve (AP = {ap_score:.4f})')
axes[1].set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

## Threshold Optimization

In [ ]:
# Evaluate performance across thresholds
thresholds = np.linspace(0.1, 0.9, 17)
recalls, precisions, f1s = [], [], []

for t in thresholds:
    preds = (y_prob_bal >= t).astype(int)
    recalls.append(recall_score(y_test, preds))
    precisions.append(precision_score(y_test, preds, zero_division=0))
    f1s.append(f1_score(y_test, preds, zero_division=0))

plt.figure(figsize=(9, 5))
plt.plot(thresholds, recalls, label='Recall', marker='o')
plt.plot(thresholds, precisions, label='Precision', marker='s')
plt.plot(thresholds, f1s, label='F1-Score', marker='^', lw=2.5)
plt.axvline(x=0.5, color='gray', linestyle='--', alpha=0.7, label='Default Threshold (0.5)')
plt.title('Performance Metrics Across Classification Thresholds', fontsize=13, fontweight='bold')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

best_idx = np.argmax(f1s)
print(f"Optimal Threshold for F1: {thresholds[best_idx]:.2f}")
print(f"Optimal F1-Score: {f1s[best_idx]:.4f} (Recall: {recalls[best_idx]:.4f}, Precision: {precisions[best_idx]:.4f})")